# Multisplit scoring and inference acceleration

Aggregate-only publication notebook generated from verified AWS receipts. It contains no raw tracking data, fitted weights, checkpoint locations, private feature transforms, or credentials.

**Current evidence:** multisplit-20 improved the recorded private score to **0.46487 RMSE** and a shared-preparation inference path reduced measured per-play latency while preserving predictions exactly on the declared parity sample.

In [1]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

scores = pd.DataFrame(
    [
        {"stage": "Previous private best", "rmse": 0.46547},
        {"stage": "Multisplit-20 private", "rmse": 0.46487},
        {"stage": "Published first-place comparator", "rmse": 0.46340},
    ]
)
display(scores)

                              stage     rmse
0             Previous private best  0.46547
1             Multisplit-20 private  0.46487
2  Published first-place comparator  0.46340

In [2]:
fig = go.Figure(
    go.Bar(
        x=scores["stage"],
        y=scores["rmse"],
        text=scores["rmse"].map(lambda value: f"{value:.5f}"),
        textposition="outside",
    )
)
fig.update_layout(
    title="Private-score progression and comparator",
    xaxis_title="Evidence",
    yaxis_title="Coordinate RMSE",
    width=900,
    height=480,
    template=None,
)
fig.show()

## Inference engineering

The accelerated path reuses prepared inputs across the 20-model ensemble. The benchmark used the same weights and observed inputs. The promoted path was bitwise-identical on **96 plays / 3,723 requested rows**; vectorized alternatives that exceeded the numerical tolerance were rejected.

In [3]:
latency = pd.DataFrame(
    [
        {"path": "Legacy repeated preparation", "seconds_per_play": 0.5411140285000329},
        {"path": "Shared preparation", "seconds_per_play": 0.11310043274988857},
    ]
)
speedup = latency.loc[0, "seconds_per_play"] / latency.loc[1, "seconds_per_play"]
latency["speedup_vs_legacy"] = latency.loc[0, "seconds_per_play"] / latency["seconds_per_play"]
display(latency)
print(f"Measured speedup: {speedup:.3f}x")

                          path  seconds_per_play  speedup_vs_legacy
0  Legacy repeated preparation          0.541114           1.000000
1           Shared preparation          0.113100           4.784367

Measured speedup: 4.784x


In [4]:
fig = go.Figure(
    go.Bar(
        x=latency["path"],
        y=latency["seconds_per_play"],
        text=latency["seconds_per_play"].map(lambda value: f"{value:.3f}s"),
        textposition="outside",
    )
)
fig.update_layout(
    title="Measured ensemble inference latency",
    xaxis_title="Execution path",
    yaxis_title="Median seconds per play",
    width=900,
    height=480,
    template=None,
)
fig.show()

## Research decisions since the prior publication

Several follow-up feature studies were deliberately **not promoted** after their control/evaluation checks failed to establish a credible improvement. The useful conclusion is methodological: preserve the strong multisplit ensemble, keep negative results, and return new feature work to the verified training contract.

In [5]:
outcomes = pd.DataFrame(
    [
        {
            "milestone": "Multisplit-20 private scoring",
            "outcome": "promoted evidence",
            "note": "0.46487 private RMSE",
        },
        {
            "milestone": "Shared-preparation inference",
            "outcome": "promoted engineering",
            "note": "4.784x measured speedup",
        },
        {
            "milestone": "Scratch feature configuration",
            "outcome": "not promoted",
            "note": "control/evaluation credibility failed",
        },
        {
            "milestone": "Normalization / continuation recovery",
            "outcome": "retired",
            "note": "control remained outside credibility gate",
        },
        {
            "milestone": "Source-recipe audit",
            "outcome": "next direction",
            "note": "restore verified training contract before new feature confirmation",
        },
    ]
)
display(outcomes)
assert abs((0.46547 - 0.46487) - 0.00060) < 1e-12
assert abs(speedup - 4.7843674453187806) < 1e-9
print("PUBLIC_SNAPSHOT_COMPLETE")

                               milestone              outcome                                               note
0          Multisplit-20 private scoring    promoted evidence                               0.46487 private RMSE
1           Shared-preparation inference  promoted engineering                            4.784x measured speedup
2          Scratch feature configuration         not promoted              control/evaluation credibility failed
3  Normalization / continuation recovery              retired          control remained outside credibility gate
4                    Source-recipe audit       next direction  restore verified training contract before new ...

PUBLIC_SNAPSHOT_COMPLETE


## Reproducibility boundary

This public notebook makes the aggregate score, timing arithmetic, decision history, and validation boundary reproducible. Private competition data, fitted weights, checkpoint locations, exact unreleased feature transforms, and credentials remain in the AWS research workspace. Kaggle is used only for the required submission surface.